In [2]:
# REGRESSION MODEL

In [10]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv(r"C:\Users\liang\Desktop\Work\Project\GAD7\GAD7_clean.csv")

In [46]:
## 1.1 CATAGORICAL DEMOGRAPHIC REGRESSION
# Replace Uknown in Age with Median For Further Analysis
reg_df = df[['score', 'gender', 'age', 'marriage', 'education', 'income']].copy()

reg_df['score'] = pd.to_numeric(reg_df['score'], errors='coerce')

reg_df['age'] = reg_df['age'].replace('Unknown', np.nan)

reg_df['age'] = pd.to_numeric(reg_df['age'], errors='coerce')

age_median = reg_df['age'].median()

reg_df['age'] = reg_df['age'].fillna(age_median)

## Regression
model = smf.ols(
    'score ~ C(gender) + age + C(marriage) + C(education) + C(income)',
    data=reg_df
).fit()

print(model.summary())

X = pd.get_dummies(
    reg_df[['gender', 'marriage', 'education', 'income']],
    drop_first=True
)

X['age'] = reg_df['age']
X = X.astype(float)
X.insert(0, 'const', 1.0)

vif_table = pd.DataFrame()
vif_table['Variable'] = X.columns
vif_table['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_table.round(4))


                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.066
Model:                            OLS   Adj. R-squared:                  0.066
Method:                 Least Squares   F-statistic:                     125.5
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:40:05   Log-Likelihood:                -94398.
No. Observations:               30000   AIC:                         1.888e+05
Df Residuals:                   29982   BIC:                         1.890e+05
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            11.7898      0.25

In [47]:
## 1.1 Result
## A multiple linear regression was performed to assess the association between demographic variables and GAD-7 score.
## The model was statistically significant, F(17, 29982) = 125.5, p < 0.001, 
## but explained only a small proportion of the variance in GAD7 score (R^2 = R^2 (adjusted) = 0.006)

## Gender :   Female participants had significantly higher GAD-7 scores than males,
## Age:       Age was significantly negatively associated with GAD-7 score.
## Marrital:  Cohabiting participants had higher scores and married participants had lower scores
## Education: Higher education levels were generally associated with lower GAD-7 scores, 
##            while the association between income and GAD-7 score was less consistent.
## VIF:       All VIF values were below 2, suggesting no serious multicollinearity.

## Interpretation
## These findings indicate that demographic factors were significantly related to GAD-7 score, but their explanatory power was limited.
## Younger age and female gender were associated with higher anxiety levels,
## and education appeared to show a protective gradient,
## with higher educational attainment linked to lower GAD-7 scores.
## Marital status and income were also associated with GAD-7 score,
## although the income pattern was less stable across categories.
## Overall, demographic variables provide a useful baseline model,
## but additional non-demographic predictors are needed to better explain variation in anxiety symptoms.

In [48]:
## 1.2 ORDINAL-TREND MODEL
reg_df = df[['score', 'gender', 'age', 'marriage', 'education', 'income']].copy()

reg_df['score'] = pd.to_numeric(reg_df['score'], errors='coerce')
reg_df['gender'] = pd.to_numeric(reg_df['gender'], errors='coerce')
reg_df['age'] = reg_df['age'].replace('Unknown', np.nan)
reg_df['age'] = pd.to_numeric(reg_df['age'], errors='coerce')
reg_df['education'] = pd.to_numeric(reg_df['education'], errors='coerce')
reg_df['income'] = pd.to_numeric(reg_df['income'], errors='coerce')
reg_df['marriage'] = pd.to_numeric(reg_df['marriage'], errors='coerce')

age_median = reg_df['age'].median()
reg_df['age'] = reg_df['age'].fillna(age_median)

reg_df = reg_df.dropna()

print("Sample size:", len(reg_df))
print("Median age used for imputation:", age_median)

# ordinal-trend model
model_trend = smf.ols(
    'score ~ gender + age + C(marriage) + education + income',
    data=reg_df
).fit()

print(model_trend.summary())

X = reg_df[['gender', 'age', 'education', 'income']].copy()

marriage_dummies = pd.get_dummies(reg_df['marriage'], prefix='marriage', drop_first=True)
X = pd.concat([X, marriage_dummies], axis=1)

X = X.astype(float)
X.insert(0, 'const', 1.0)

vif_table = pd.DataFrame()
vif_table['Variable'] = X.columns
vif_table['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_table.round(4))

Sample size: 30000
Median age used for imputation: 24.0
                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.065
Method:                 Least Squares   F-statistic:                     260.0
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:40:11   Log-Likelihood:                -94424.
No. Observations:               30000   AIC:                         1.889e+05
Df Residuals:                   29991   BIC:                         1.889e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

In [49]:
## 1.2 Result
## The ordinal-trend model yielded results largely consistent with the categorical model.
## Gender, age, and marital status remained significant in the same directions.
## Unlike the categorical model, education and income were modeled as ordered variables,
## and both showed significant negative trend effects,
## indicating that higher education and higher income levels were associated with lower GAD-7 scores.
## Model fit was nearly unchanged. (R^2 = 0.065) 

## Interpretation
## Overall, the two models produced very similar conclusions.
## Their main similarity is that the core associations for gender, age, and marital status were stable across specifications.
## Their main difference is that the ordinal-trend model captured education and income as overall gradients rather than category-specific contrasts,
## Since the explanatory power was almost identical to that of the categorical model,
## the ordinal version supports the presence of a general socioeconomic gradient in anxiety but does not substantially improve model performance.

In [42]:
## 2. REGRESSION MODEL WITH THEME-LEVEL COPING VARIABLES

## 2.1 Prepare baseline dataframe
reg_df = df[['score', 'gender', 'age', 'marriage', 'education', 'income']].copy()

reg_df['score'] = pd.to_numeric(reg_df['score'], errors='coerce')
reg_df['gender'] = pd.to_numeric(reg_df['gender'], errors='coerce')
reg_df['age'] = reg_df['age'].replace('Unknown', np.nan)
reg_df['age'] = pd.to_numeric(reg_df['age'], errors='coerce')
reg_df['marriage'] = pd.to_numeric(reg_df['marriage'], errors='coerce')
reg_df['education'] = pd.to_numeric(reg_df['education'], errors='coerce')
reg_df['income'] = pd.to_numeric(reg_df['income'], errors='coerce')

age_median = reg_df['age'].median()
reg_df['age'] = reg_df['age'].fillna(age_median)


# Add coping item columns
coping_cols = [
    'q500_0','q500_1','q500_2','q500_3','q500_4','q500_5','q500_6','q500_7',
    'q510_0','q510_1','q510_2','q510_3','q510_4','q510_5','q510_6','q510_7','q510_8',
    'q520_0','q520_1','q520_2','q520_3','q520_4','q520_5','q520_6','q520_7','q520_8','q520_9',
    'q530_0','q530_1','q530_2','q530_3','q530_4','q530_5','q530_6',
    'q540_0','q540_1','q540_2','q540_3','q540_4','q540_5',
    'q560_0','q560_1','q560_2','q560_3','q560_4','q560_5','q560_6','q560_7','q560_8'
]

for col in coping_cols:
    reg_df[col] = pd.to_numeric(df[col], errors='coerce')

reg_df[coping_cols] = reg_df[coping_cols].fillna(0)


## 2.2 Build coping theme variables

# emotional release
reg_df['emotional_release'] = reg_df[['q500_0', 'q500_1']].max(axis=1)

# maladaptive coping
reg_df['maladaptive_coping'] = reg_df[['q500_2', 'q500_3', 'q500_4', 'q500_6']].max(axis=1)

# distraction coping
reg_df['distraction_coping'] = reg_df[
    ['q510_0', 'q510_1', 'q510_2', 'q510_3', 'q510_4', 'q510_6', 'q510_7', 'q510_8',
     'q520_4', 'q520_5', 'q500_5']
].max(axis=1)

# problem-focused coping
reg_df['problem_focused_coping'] = reg_df[['q520_0', 'q520_1', 'q520_3', 'q520_7']].max(axis=1)

# help-seeking coping
reg_df['help_seeking_coping'] = reg_df[['q530_0', 'q530_2', 'q530_6']].max(axis=1)

# self-soothing coping
reg_df['self_soothing_coping'] = reg_df[
    ['q520_8', 'q520_9', 'q540_0', 'q540_1', 'q540_2', 'q540_3', 'q540_4', 'q540_5']
].max(axis=1)

# exercise coping
exercise_cols = [
    'q500_7', 'q520_2', 'q530_4',
    'q560_0','q560_1','q560_2','q560_3','q560_4','q560_5','q560_6','q560_7','q560_8'
]
reg_df['exercise_coping'] = reg_df[exercise_cols].max(axis=1)

## 2.3 Final regression dataframe
model_df = reg_df[
    ['score', 'gender', 'age', 'marriage', 'education', 'income',
     'emotional_release', 'maladaptive_coping', 'distraction_coping',
     'problem_focused_coping', 'help_seeking_coping',
     'self_soothing_coping', 'exercise_coping']
].dropna()

print("Sample size:", len(model_df))
print("Median age used for imputation:", age_median)

print(model_df[
    ['emotional_release', 'maladaptive_coping', 'distraction_coping',
     'problem_focused_coping', 'help_seeking_coping',
     'self_soothing_coping', 'exercise_coping']
].mean().round(4))


## 2.5 Fit Model A
model_a = smf.ols(
    'score ~ gender + age + C(marriage) + education + income'
    ' + emotional_release + maladaptive_coping + distraction_coping'
    ' + problem_focused_coping + help_seeking_coping'
    ' + self_soothing_coping + exercise_coping',
    data=model_df
).fit()

print(model_a.summary())

## 2.6 VIF
X = model_df[
    ['gender', 'age', 'education', 'income',
     'emotional_release', 'maladaptive_coping', 'distraction_coping',
     'problem_focused_coping', 'help_seeking_coping',
     'self_soothing_coping', 'exercise_coping']
].copy()

marriage_dummies = pd.get_dummies(model_df['marriage'], prefix='marriage', drop_first=True)
X = pd.concat([X, marriage_dummies], axis=1)

X = X.astype(float)
X.insert(0, 'const', 1.0)

vif_table = pd.DataFrame({
    'Variable': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

print(vif_table.round(4))

Sample size: 30000
Median age used for imputation: 24.0
emotional_release         0.3904
maladaptive_coping        0.4976
distraction_coping        0.8896
problem_focused_coping    0.4609
help_seeking_coping       0.6059
self_soothing_coping      0.9072
exercise_coping           0.4173
dtype: float64
                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.137
Model:                            OLS   Adj. R-squared:                  0.137
Method:                 Least Squares   F-statistic:                     318.2
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:33:16   Log-Likelihood:                -93214.
No. Observations:               30000   AIC:                         1.865e+05
Df Residuals:                   29984   BIC:                         1.866e+05
Df Model:                          15                             

In [44]:
## 2. Result
## A multiple linear regression model including both demographic variables and coping-strategy themes was statistically significant
## F(15, 29984) = 318.2, p < 0.001, with R^2 = R^2 adjusted = 0.137.
## Compared with the demographic-only models, model fit improved notably.
## Emotional release, maladaptive coping, and problem-focused coping were positively associated with GAD-7 score,
## whereas distraction coping, help-seeking coping, self-soothing coping, and exercise coping were negatively associated with GAD-7 score. (all p < 0.001)
## Among demographic factors, gender, age, education, and married status remained significant, while income was no longer significant. 
## All VIF values were below 2.

## Interpretation
## These results indicate that coping-strategy themes contribute substantially to explaining variation in GAD-7 score beyond demographic factors alone.
## Maladaptive coping showed the strongest positive association with anxiety, 
## while distraction, help-seeking, self-soothing, and exercise coping were associated with lower anxiety scores.
## The demographic effects remained generally consistent with earlier models,
## but the improved R^2 suggests that coping styles are more informative correlates of GAD-7 score than demographic characteristics alone.

In [52]:
## 3. REGRESSION MODEL WITH SELECTED ITEM-LEVEL COPING VARIABLES

# 3.1 Add selected coping items
selected_cols = [
    'q500_0', 'q500_1', 'q500_2', 'q500_4', 'q500_6',
    'q510_6', 'q510_7', 'q510_8',
    'q520_0', 'q520_2', 'q520_3',
    'q530_0', 'q530_2', 'q530_4',
    'q540_2', 'q540_3', 'q540_4',
    'q560_0','q560_1','q560_2','q560_3','q560_4','q560_5','q560_6','q560_7','q560_8'
]

for col in selected_cols:
    reg_df[col] = pd.to_numeric(df[col], errors='coerce')

reg_df[selected_cols] = reg_df[selected_cols].fillna(0)


# 3.2 Exercise type count
reg_df['exercise_type_count'] = reg_df[
    ['q560_0','q560_1','q560_2','q560_3','q560_4','q560_5','q560_6','q560_7','q560_8']
].sum(axis=1)


# 3.3 Final dataframe
model_b_df = reg_df[
    ['score', 'gender', 'age', 'marriage', 'education', 'income',
     'q500_0', 'q500_1', 'q500_2', 'q500_4', 'q500_6',
     'q510_6', 'q510_7', 'q510_8',
     'q520_0', 'q520_2', 'q520_3',
     'q530_0', 'q530_2', 'q530_4',
     'q540_2', 'q540_3', 'q540_4',
     'exercise_type_count']
].dropna()

print("\nSelected item prevalence:")
print(model_b_df[
    ['q500_0', 'q500_1', 'q500_2', 'q500_4', 'q500_6',
     'q510_6', 'q510_7', 'q510_8',
     'q520_0', 'q520_2', 'q520_3',
     'q530_0', 'q530_2', 'q530_4',
     'q540_2', 'q540_3', 'q540_4']
].mean().round(4))

print("\nExercise type count:")
print(model_b_df['exercise_type_count'].describe())


# 3.4 Fit Model B
model_b = smf.ols(
    'score ~ gender + age + C(marriage) + education + income'
    ' + q500_0 + q500_1 + q500_2 + q500_4 + q500_6'
    ' + q510_6 + q510_7 + q510_8'
    ' + q520_0 + q520_2 + q520_3'
    ' + q530_0 + q530_2 + q530_4'
    ' + q540_2 + q540_3 + q540_4'
    ' + exercise_type_count',
    data=model_b_df
).fit()

print(model_b.summary())


Selected item prevalence:
q500_0    0.3338
q500_1    0.1282
q500_2    0.3007
q500_4    0.1072
q500_6    0.1956
q510_6    0.3749
q510_7    0.1576
q510_8    0.2559
q520_0    0.3137
q520_2    0.1930
q520_3    0.1214
q530_0    0.5628
q530_2    0.0730
q530_4    0.1008
q540_2    0.6238
q540_3    0.2831
q540_4    0.3537
dtype: float64

Exercise type count:
count    30000.000000
mean         0.765100
std          1.249923
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          9.000000
Name: exercise_type_count, dtype: float64
                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.185
Method:                 Least Squares   F-statistic:                     263.3
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:44

In [54]:
## 3. Result
## A multiple linear regression model including demographic variables and selected item-level coping behaviors was statistically significant,
## F(26, 29973) = 263.3, p < 0.001, with R^2 = 0.186 and R^2 adjusted = 0.185.
## Compared with the theme-level coping model, model fit improved further.
## Positive associations with GAD-7 score were observed for 
## crying, venting emotions toward family, self-harm, alcohol use, meditation, and seeking professional help, 
## while gaming, travel, meeting friends, problem-solving, talking with family or friends,
## group exercise, sleep, shopping, enjoying food, and exercise type
## count were negatively associated with GAD-7 score.
## Shouting and exercising alone were not significant.
## Gender, age, education, and married status remained significant, whereas income did not.

## Interpretation
## This model indicates that specific coping behaviors explain more variation in GAD-7 score than broader coping themes.
## Behaviors such as self-harm, alcohol use, emotional venting, and professional help-seeking were associated with higher anxiety scores,
## whereas social interaction, restorative activities, and more diverse exercise engagement were associated with lower anxiety scores.
## Compared with Model 2, Model 3 provides a more detailed picture of which specific coping behaviors may be driving the broader theme-level associations.